In [ ]:
%pip install scrubadub sklearn_crfsuite

import os
import tensorflow as tf
import numpy as np
import nltk
import pickle
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import Tokenizer
from typing import Set
import re
import json
import scrubadub
import joblib
import sklearn_crfsuite

# Download NLTK stopwords if necessary
nltk.download('stopwords')
from nltk.corpus import stopwords

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Import necessary libraries
from pyspark.sql import SparkSession
import pandas as pd

# Set up Spark session
spark = SparkSession.builder \
    .appName("Verbatims") \
    .getOrCreate()

# Define path to your data in Azure Data Lake Storage
data_path = "Verbatim and Topics v3.xlsx"

# Load data into a Spark DataFrame
df = spark.read.format("com.crealytics.spark.excel").option("header", "true").load(data_path)

# Load Spark DF to Pandas DF
pandas_df = df.toPandas()

display(pandas_df)

df = pandas_df.copy()  # create a copy to avoid modifying the original DataFrame

df.columns = ['o_comment', 'topic']

In [ ]:
# Implementing stopwords
stopwords = set(stopwords.words('english'))
custom_stop_words = { 'water'}
stop_words = stopwords.union(custom_stop_words)

In [ ]:
def preprocess_text(text):
    text = text.lower().strip()  # lowercasing and stripping whitespace
    text = scrubadub.clean(text) # removing PII
    
    tokens = text.split()
    filtered_tokens = [token for token in tokens if token not in stop_words] # removing stopwords
    text = " ".join(filtered_tokens)
    return text

# Preprocess comments and add special tokens for start and end of sentence.
# Also, we use a special token (<oov>) for out-of-vocabulary words.
df['comment'] = df['o_comment'].apply(preprocess_text)
df['comment'] = df['comment'].apply(lambda x: "<start> " + x + " <end>")

# Initialize the tokenizer with a token for OOV words
tokenizer = Tokenizer(oov_token="<oov>")
tokenizer.fit_on_texts(df['comment'])
word_index = tokenizer.word_index
print(f"Size of vocabulary: {len(word_index)}")

Size of vocabulary: 7502


In [ ]:
crf_model = joblib.load("machi_v2.pkl")

In [ ]:
def tokenize_with_offsets(text):
    # Define a regex pattern to match words and non-whitespace characters
    pattern = r"\w+|[^\w\s]"
    # Use regex to find all matches in the text and return them with their start and end positions
    return [(m.group(), m.start(), m.end()) for m in re.finditer(pattern, text)]

def word2features(tokens, i):
    # Get the current word from the tokens list
    w = tokens[i]
    # Create a feature dictionary for the current word
    feats = {
        "bias": 1.0,  # Bias feature for the model
        "word.lower": w.lower(),  # Lowercase version of the word
        "word[-3:]": w[-3:],  # Last three characters of the word
        "word.isupper": w.isupper(),  # Check if the word is uppercase
        "word.istitle": w.istitle(),  # Check if the word is title case
        "word.isdigit": w.isdigit(),  # Check if the word is a digit
    }
    # If not the first word, add features for the previous word
    if i > 0:
        p = tokens[i - 1]
        feats.update({
            "-1:word.lower": p.lower(),  # Lowercase of the previous word
            "-1:word.istitle": p.istitle(),  # Title case check for the previous word
            "-1:word.isupper": p.isupper(),  # Uppercase check for the previous word
        })
    else:
        feats["BOS"] = True  # Beginning of sentence feature
    # If not the last word, add features for the next word
    if i < len(tokens) - 1:
        n = tokens[i + 1]
        feats.update({
            "+1:word.lower": n.lower(),  # Lowercase of the next word
            "+1:word.istitle": n.istitle(),  # Title case check for the next word
            "+1:word.isupper": n.isupper(),  # Uppercase check for the next word
        })
    else:
        feats["EOS"] = True  # End of sentence feature
    return feats  # Return the feature dictionary

def sent2features(tokens):
    # Generate features for each token in the sentence
    return [word2features(tokens, i) for i in range(len(tokens))]

In [ ]:
# Australia Post street types (code + full names)
STREET_TYPES = [
    "ALLY","ALLEY","ALWY","ALLEYWAY","AMBL","AMBLE","ANCG","ANCHORAGE","APP","APPROACH",
    "ARC","ARCADE","ART","ARTERY","AVE","AVENUE","BASN","BASIN","BCH","BEACH","BEND","BEND",
    "BLK","BLOCK","BVD","BOULEVARD","BRCE","BRACE","BRAE","BRAE","BRK","BREAK","BDGE","BRIDGE",
    "BDWY","BROADWAY","BROW","BROW","BYPA","BYPASS","BYWY","BYWAY","CAUS","CAUSEWAY","CTR","CENTRE",
    "CNWY","CENTREWAY","CH","CHASE","CIR","CIRCLE","CLT","CIRCLET","CCT","CIRCUIT","CRCS","CIRCUS",
    "CL","CLOSE","CLDE","COLONNADE","CMMN","COMMON","CON","CONCOURSE","CPS","COPSE","CNR","CORNER",
    "CSO","CORSO","CT","COURT","CTYD","COURTYARD","COVE","COVE","CRES","CRESCENT","CRST","CREST",
    "CRSS","CROSS","CRSG","CROSSING","CRD","CROSSROAD","COWY","CROSSWAY","CUWY","CRUISEWAY",
    "CDS","CUL-DE-SAC","CTTG","CUTTING","DALE","DALE","DELL","DELL","DEVN","DEVIATION","DIP","DIP",
    "DSTR","DISTRIBUTOR","DR","DRIVE","DRWY","DRIVEWAY","EDGE","EDGE","ELB","ELBOW","END","END",
    "ENT","ENTRANCE","ESP","ESPLANADE","EST","ESTATE","EXP","EXPRESSWAY","EXTN","EXTENSION",
    "FAWY","FAIRWAY","FTRK","FIRE","FITR","FIRETRAIL","FLAT","FLAT","FOLW","FOLLOW","FTWY","FOOTWAY",
    "FSHR","FORESHORE","FORM","FORMATION","FWY","FREEWAY","FRNT","FRONT","FRTG","FRONTAGE","GAP","GAP",
    "GDN","GARDEN","GTE","GATE","GDNS","GARDENS","GTES","GATES","GLD","GLADE","GLEN","GLEN","GRA","GRANGE",
    "GRN","GREEN","GRND","GROUND","GR","GROVE","GLY","GULLY","HTS","HEIGHTS","HRD","HIGHROAD","HWY","HIGHWAY",
    "HILL","HILL","INTG","INTERCHANGE","INTN","INTERSECTION","JNC","JUNCTION","KEY","KEY","LDG","LANDING",
    "LANE","LANE","LNWY","LANEWAY","LEES","LEES","LINE","LINE","LINK","LINK","LT","LITTLE","LKT","LOOKOUT",
    "LOOP","LOOP","LWR","LOWER","MALL","MALL","MNDR","MEANDER","MEW","MEW","MEWS","MEWS","MWY","MOTORWAY",
    "MT","MOUNT","NOOK","NOOK","OTLK","OUTLOOK","PDE","PARADE","PARK","PARK","PKLD","PARKLANDS","PKWY","PARKWAY",
    "PART","PART","PASS","PASS","PATH","PATH","PHWY","PATHWAY","PIAZ","PIAZZA","PL","PLACE","PLAT","PLATEAU",
    "PLZA","PLAZA","PKT","POCKET","PNT","POINT","PORT","PORT","PROM","PROMENADE","QUAD","QUAD","QDGL","QUADRANGLE",
    "QDRT","QUADRANT","QY","QUAY","QYS","QUAYS","RMBL","RAMBLE","RAMP","RAMP","RNGE","RANGE","RCH","REACH",
    "RES","RESERVE","REST","REST","RTT","RETREAT","RIDE","RIDE","RDGE","RIDGE","RGWY","RIDGEWAY","ROWY","RIGHT",
    "RING","RING","RISE","RISE","RVR","RIVER","RVWY","RIVERWAY","RVRA","RIVIERA","RD","ROAD","RDS","ROADS",
    "RDSD","ROADSIDE","RDWY","ROADWAY","RNDE","RONDE","RSBL","ROSEBOWL","RTY","ROTARY","RND","ROUND","RTE","ROUTE",
    "ROW","ROW","RUE","RUE","RUN","RUN","SWY","SERVICE","SDNG","SIDING","SLPE","SLOPE","SND","SOUND","SPUR","SPUR",
    "SQ","SQUARE","STRS","STAIRS","SHWY","STATE","STPS","STEPS","STRA","STRAND","ST","STREET","STRP","STRIP",
    "SBWY","SUBWAY","TARN","TARN","TCE","TERRACE","THOR","THOROUGHFARE","TLWY","TOLLWAY","TOP","TOP","TOR","TOR",
    "TWRS","TOWERS","TRK","TRACK","TRL","TRAIL","TRLR","TRAILER","TRI","TRIANGLE","TKWY","TRUNKWAY","TURN","TURN",
    "UPAS","UNDERPASS","UPR","UPPER","VALE","VALE","VDCT","VIADUCT","VIEW","VIEW","VLLS","VILLAS","VSTA","VISTA",
    "WADE","WADE","WALK","WALK","WKWY","WALKWAY","WAY","WAY","WHRF","WHARF","WYND","WYND","YARD","YARD"
]

STREET_PATTERN = re.compile(
    r"\b\d+\s+[A-Za-z0-9\s]+\s+(?:" + "|".join(STREET_TYPES) + r")\b",
    flags=re.I
)

In [ ]:
def redact_crf(text: str) -> str:
    """
    1) Early regex‐based redaction on the raw text
    2) CRF‐based redaction on the remainder
    """
    # Remove unreadable characters
    text = re.sub(r'[^\x00-\x7F]+', '', text)

    # ── 1) Early regex fallbacks on raw text ────────────────────────────
    # email addresses (remain contiguous so this catches them)
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "[redacted]", text
    )
    # mobile numbers: 4-3-3 format
    text = re.sub(r"\b\d{4}\s\d{3}\s\d{3}\b", "[redacted]", text)
    # street addresses: "12 Elm Avenue"
    text = STREET_PATTERN.sub("[redacted]", text)
    # city/state/postcode: "Melbourne VIC 3004"
    text = re.sub(
        r"\b[A-Za-z ]+\s+(VIC|NSW|QLD|SA|WA|TAS|NT|ACT)\s*\d{4}\b",
        "[redacted]", text, flags=re.I
    )
    # 7-digit numbers
    text = re.sub(r"\b\d{7}\b", "[redacted]", text)
    # final numeric catch-alls for medicare/concession/license
    for pat in [
        r"\b\d{4}\s\d{5}\s\d\b",
        r"\b\d{3}\s\d{3}\s\d{3}[A-Za-z]\b",
        r"\b\d{9}\b"
    ]:
        text = re.sub(pat, "[redacted]", text)

    # ── 2) CRF-based redaction on what's left ────────────────────────────
    toks   = tokenize_with_offsets(text)
    words  = [w for w, _, _ in toks]
    feats  = sent2features(words)
    tags   = crf_model.predict_single(feats)

    out, i = [], 0
    while i < len(words):
        if tags[i].startswith("B-"):
            # skip the whole entity
            while i < len(words) and tags[i].startswith(("B-","I-")):
                i += 1
            out.append("[redacted]")
        else:
            out.append(words[i])
            i += 1

    return " ".join(out)

In [ ]:
# ---------------------------------------------
# Preprocessing Setup (HIMARI)
# ---------------------------------------------
def himari_preprocess(text: str, case_origin: str, stop_words: Set[str]) -> str:
    """
    1) Extract body by case_origin  
    2) lowercase + strip  
    3) redact PII via CRF (redact_crf)  
    4) remove stopwords  
    5) wrap with <start> / <end>
    """
    # --- Case‐origin extraction ---
    if case_origin == "Email":
        # Remove classic headers (From/Sent/To/Subject lines)
        text = re.sub(r'^(?:From|Sent|To|Subject):.*\r?\n', '', text, flags=re.I|re.M)
        # If an inline reply appears, cut there
        m = re.search(r'(^On .+ wrote:)', text, flags=re.I|re.M)
        if m:
            text = text[:m.start()]
        # Drop forwarded‐message markers
        text = re.split(r'-----Original Message-----', text, flags=re.I)[0]

    elif case_origin == "Web":
        m = re.search(r"Case Description:(.*?)(?=First Name:)", text, flags=re.S|re.I)
        if m:
            text = m.group(1)

    # --- Normalize ---
    text = text.lower().strip()

    # --- PII Redaction ---
    text = redact_crf(text)

    # --- Stopword removal ---
    toks     = text.split()
    filtered = [t for t in toks if t not in stop_words]

    # --- Truncate so that after adding <start> and <end> we never exceed MAX_SEQUENCE_LENGTH ---
    # reserve 2 spots for the special tokens
    max_body_tokens = MAX_SEQUENCE_LENGTH - 2
    if len(filtered) > max_body_tokens:
        filtered = filtered[:max_body_tokens]

    # --- Wrap and return ---
    return "<start> " + " ".join(filtered) + " <end>"

In [ ]:
# Set model path
model_path = 'hikari_v2.keras'   # HIKARI model

# Load HIKARI model
if os.path.exists(model_path):
    if os.path.isfile(model_path):
        try:
            hikari_model = load_model(model_path, compile=False)
            print("HIKARI model loaded successfully.")
            print("Model type:", type(hikari_model))  # Should be Sequential or Functional
            print("Input shape:", hikari_model.input_shape)
        except Exception as e:
            print(f"Error loading HIKARI model: {e}")
    else:
        print(f"{model_path} is a directory, not a file. Expected a .keras single file.")
else:
    print(f"Model file not found at: {model_path}")

HIKARI model loaded successfully.
Model type: <class 'keras.src.engine.sequential.Sequential'>
Input shape: (None, 196)


In [ ]:
from tensorflow.keras.backend import int_shape

# once hikari_model is loaded...
# int_shape returns a tuple like (None, 196)
MAX_SEQUENCE_LENGTH = int_shape(hikari_model.input)[1]
print("Auto-detected max sequence length:", MAX_SEQUENCE_LENGTH)

In [ ]:
# ---------------------------------------------
#  Prediction Function
# ---------------------------------------------

def predict_drivers_from_comment(
    raw_comment: str,
    case_origin: str = "Other",
    threshold: float = 0.5
):
    """
    Preprocess and predict drivers from a raw customer comment.
    """
    # Preprocess (now passing both origin & stop_words)
    cleaned_comment = himari_preprocess(raw_comment, case_origin, stop_words)

    # Convert to sequence
    sequence = tokenizer.texts_to_sequences([cleaned_comment])
    padded   = pad_sequences(sequence, maxlen=196, padding='post', truncating='post')

    # Predict
    probabilities = hikari_model.predict(padded, verbose=0)[0]

    # Interpret results
    predicted_drivers = []
    topics = [
        'Customer Service','Digital','Online Experience','Outages and Faults',
        'Process','Reputation','Sustainability','Trust',
        'Value for Money','Vulnerability & FDV'
    ]
    for idx, prob in enumerate(probabilities):
        if prob >= threshold:
            predicted_drivers.append((topics[idx], round(float(prob), 3)))

    # Sort by probability
    predicted_drivers.sort(key=lambda x: x[1], reverse=True)

    return predicted_drivers, cleaned_comment

In [ ]:
# Define path to your data in Azure Data Lake Storage
data_path = "Complaints v4.2.xlsx"

# Load data into a Spark DataFrame
com_df = spark.read.format("com.crealytics.spark.excel").option("header", "true").load(data_path)

# Load Spark DF to Pandas DF
com_pandas_df = com_df.toPandas()

complaints_df = com_pandas_df.copy()  # create a copy to avoid modifying the original DataFrame

In [ ]:
# create a new column with only three buckets: Email, Web, Other
complaints_df['origin'] = (
    df['Case Origin']
      .str.lower()
      .apply(lambda s: 'Email' if s == 'email' else 'Web' if s == 'web' else 'Other')
)

In [ ]:
def process_cases(df: pd.DataFrame, threshold: float = 0.5) -> pd.DataFrame:
    """
    Input:
      df with columns ['Case Number', 'Description', 'Case Origin']
    Returns:
      DataFrame with ['Case Number', 'clean_comment', 'drivers']
    """
    results = []
    
    for _, row in df.iterrows():
        case_num   = row['Case Number']
        raw_text   = row['Description']
        origin     = row['origin']
        
        # --- run your combined preprocessing + prediction ---
        drivers_probs, clean_comment = predict_drivers_from_comment(
            raw_comment = raw_text,
            case_origin = origin,
            threshold   = threshold
        )
        
        # drivers_probs is a list of (driver_name, prob)
        drivers = [drv for drv, _ in drivers_probs]
        drivers_str = ",".join(drivers) if drivers else "None"
        
        results.append({
            'Case Number'  : case_num,
            'clean_comment': clean_comment,
            'drivers'      : drivers_str
        })
    
    return pd.DataFrame(results)

In [ ]:
output_df = process_cases(complaints_df, threshold=0.5)